# storm-report → GitHub 上传器（Colab）

把本地打包好的 `storm-report-github.zip` 上传到 Google Colab，并推送到你的 GitHub 仓库。

## 前置条件

1. 一个 GitHub **Personal Access Token**：
   - 经典 token：勾选 `repo` 权限
   - 或 fine-grained token：`Contents` 读写 + `Metadata` 只读，并对目标仓库授权
   - 申请地址：GitHub → Settings → Developer settings → Personal access tokens
2. 本地的 `storm-report-github.zip`（本仓库的打包文件）

## 使用步骤

按顺序执行下面每一格即可。Token 通过 `getpass` 交互输入，**不会写入文件、不会出现在输出里**。

In [ ]:
# ① 安装依赖 & 配置
!pip install -q PyGithub

from getpass import getpass
from github import Github

TOKEN = getpass("粘贴 GitHub Personal Access Token（需 repo 权限）: ")

REPO_NAME   = "storm-report"          # 想要的仓库名，可改
DESCRIPTION = "基于斯坦福 STORM 方法的行业/市场研究报告 skill"
PRIVATE     = False                   # True = 私有仓库

g = Github(TOKEN)
GITHUB_USER = g.get_user().login
print("GitHub 用户:", GITHUB_USER, "| 目标仓库:", REPO_NAME)

## ② 上传并解压打包文件

运行下一格，选择本地的 `storm-report-github.zip`。上传完成后会自动解压到 `/content/storm-report-github`。

In [ ]:
from google.colab import files
import zipfile, io, os

uploaded = files.upload()   # 选择 storm-report-github.zip

for name, data in uploaded.items():
    if name.lower().endswith(".zip"):
        zipfile.ZipFile(io.BytesIO(data)).extractall("/content/")
        print("已解压:", name)

ROOT = "/content/storm-report-github"
print("仓库根目录:", ROOT)
print(os.listdir(ROOT))

## ③ 推送到 GitHub

仓库不存在则自动创建；已存在则逐文件更新（按文件路径判断新增 / 更新）。

In [ ]:
from github import UnknownObjectException
import os

user = g.get_user()

try:
    repo = user.create_repo(REPO_NAME, private=PRIVATE, description=DESCRIPTION)
    print("已创建仓库:", repo.html_url)
except Exception:
    repo = user.get_repo(REPO_NAME)
    print("仓库已存在，改为更新:", repo.html_url)

for dirpath, _, filenames in os.walk(ROOT):
    for fn in filenames:
        full = os.path.join(dirpath, fn)
        rel = os.path.relpath(full, ROOT).replace(os.sep, "/")
        if rel.startswith(".git/") or "__pycache__" in rel:
            continue
        with open(full, encoding="utf-8") as f:
            content = f.read()
        try:
            item = repo.get_contents(rel)
            repo.update_file(rel, "update " + rel, content, item.sha)
            print("更新:", rel)
        except UnknownObjectException:
            repo.create_file(rel, "add " + rel, content)
            print("新增:", rel)

print("全部完成 ->", repo.html_url)

---

## 备选方案：用 git CLI 推送

如果你更习惯 git 命令行，可改用下面这一格（**需先执行第 ① 格**以取得 `TOKEN` / `GITHUB_USER` / `REPO_NAME`）。

注意：git 方式**不会自动创建仓库**，请先在 GitHub 网页上建一个**空仓库**（不要勾选 README / .gitignore），再运行。

In [ ]:
# 备选：git CLI 方式（先确保已在 GitHub 建好空仓库）
!cd /content/storm-report-github && git init -q && git add -A \
  && git -c user.email=colab@example.com -c user.name=colab commit -q -m "init: storm-report skill" \
  && git branch -M main \
  && (git remote remove origin || true) \
  && git remote add origin https://$TOKEN@github.com/$GITHUB_USER/$REPO_NAME.git \
  && git push -u origin main